# 05 — dbt: Production-Grade Transforms

In notebook 02 we built the SILVER/GOLD layer using **Dynamic Tables** — great for prototyping.
Now let's see what the same transforms look like in **production** using dbt Projects on Snowflake.

This notebook:
1. Creates a dbt Project object from pre-staged files
2. Executes `dbt run` to materialise models
3. Runs `dbt test` for data quality
4. Shows how to schedule for production


## Step 1: Deploy the dbt Project Object

This creates a native Snowflake object from the dbt source files.
You can do this from Workspaces UI (recommended) or via SQL.

In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

-- Stage already populated by the deploy script
-- Files are at @PUBLIC.DBT_STAGE/hydrab_fleet/
CREATE STAGE IF NOT EXISTS PUBLIC.DBT_STAGE;

### Upload dbt files to stage

In Workspaces, you can deploy directly from the UI.
Alternatively, upload the `dbt_project/` folder to the stage:

In [ ]:
-- Create the dbt project object from pre-staged files
CREATE OR REPLACE DBT PROJECT GOLD.HYDRAB_FLEET_DBT
  FROM '@PUBLIC.DBT_STAGE/hydrab_fleet'
  COMMENT = 'HydraB Fleet dbt project - production transforms';

## Step 2: Execute the dbt Project

This runs `dbt run` — compiling models and materialising tables in SILVER/GOLD.

In [ ]:
-- Run all models
EXECUTE DBT PROJECT GOLD.HYDRAB_FLEET_DBT
  ARGS = 'run'
  TARGET_SCHEMA = 'GOLD'
  WAREHOUSE = 'HYDRAB_HOL_WH';

In [ ]:
-- Run tests to validate data quality
EXECUTE DBT PROJECT GOLD.HYDRAB_FLEET_DBT
  ARGS = 'test'
  TARGET_SCHEMA = 'GOLD'
  WAREHOUSE = 'HYDRAB_HOL_WH';

## Step 3: View the DAG in Workspaces

Go to **Projects > Workspaces** in Snowsight:
1. Open your workspace containing the dbt files
2. Select the project and profile
3. Run **Compile**
4. Click the **DAG** tab below the editor

You'll see the full lineage: sources → staging → marts

## Step 4: Schedule with a Task

For production, schedule the dbt project to run on a cadence:

In [ ]:
-- Schedule dbt run every hour
CREATE OR REPLACE TASK GOLD.DBT_HOURLY_RUN
  WAREHOUSE = HYDRAB_HOL_WH
  SCHEDULE = 'USING CRON 0 * * * * UTC'
AS
  EXECUTE DBT PROJECT GOLD.HYDRAB_FLEET_DBT
    ARGS = 'run'
    TARGET_SCHEMA = 'GOLD';

-- Activate the task
ALTER TASK GOLD.DBT_HOURLY_RUN RESUME;

In [ ]:
-- Verify the task was created
SHOW TASKS IN SCHEMA GOLD;

## Step 5: Verify Results

In [ ]:
-- Check that dbt materialised the tables
SHOW TABLES IN SCHEMA GOLD;

-- Query the mart
SELECT * FROM GOLD.DIM_VEHICLE LIMIT 10;

In [ ]:
-- View dbt project metadata
SHOW DBT PROJECTS IN SCHEMA GOLD;
DESCRIBE DBT PROJECT GOLD.HYDRAB_FLEET_DBT;

## Summary: Dynamic Tables vs dbt

| | Dynamic Tables (NB 03) | dbt Project (NB 05) |
|---|---|---|
| **Speed to prototype** | Instant | Need project structure |
| **Refresh** | Automatic (TARGET_LAG) | Scheduled via Task |
| **Testing** | Manual | Built-in (schema.yml) |
| **Lineage/DAG** | Snowsight object graph | dbt DAG in Workspace |
| **Version control** | N/A | Git-integrated |
| **Documentation** | Comments | Auto-generated docs |
| **Best for** | Real-time dashboards | Production pipelines |

Both are valid — use Dynamic Tables for real-time needs, dbt for governed, tested pipelines.

---

## Cleanup

In [ ]:
-- Suspend the task (don't leave it running after the lab)
ALTER TASK GOLD.DBT_HOURLY_RUN SUSPEND;

-- Optional: drop the dbt project
-- DROP DBT PROJECT IF EXISTS GOLD.HYDRAB_FLEET_DBT;